# Diagnóstico y evidencia de decisiones — docs2llm

Este notebook explica, paso a paso, las decisiones más importantes detrás del pipeline (`src/docs2llm/`): qué se decidió, qué otras opciones se consideraron y por qué se descartaron, y qué evidencia real de `docs_raw/` (la documentación cruda de los 10 proyectos) respalda cada elección.

In [1]:
# Setup: imports y objetos reusados en el resto del notebook.

# autoreload: si se edita un archivo de src/docs2llm/ mientras el kernel de
# este notebook sigue vivo, Python NO relee ese archivo solo -- reusa el
# modulo que ya importo en memoria (sys.modules), aunque el .py en disco haya
# cambiado. %autoreload 2 hace que IPython vuelva a importar el codigo editado
# antes de cada celda, sin tener que reiniciar el kernel a mano cada vez.
%load_ext autoreload
%autoreload 2

import hashlib
import json
import re
from collections import Counter, defaultdict

import pandas as pd

from docs2llm.config import DOCS_RAW_DIR, REPO_ROOT, load_canonical_sources, load_pipeline_config
from docs2llm.discovery import discover_source_files

sources = load_canonical_sources()
pipeline_cfg = load_pipeline_config()
files = discover_source_files(sources)

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"Proyectos resueltos: {len(sources)}")
print(f"Archivos fuente descubiertos (tras filtrar navegacion/metadata): {len(files)}")

[discovery] 2p-revenue-optimizer-api (latest): 4 archivos encontrados
[discovery] catalog-portfolio-api (latest): 8 archivos encontrados
[discovery] demand-forecast-docs (latest): 4 archivos encontrados
[discovery] order-workflow-api (5.6.2-hotfix-1): 8 archivos encontrados
[discovery] payment-promise-gateway (latest): 1 archivos encontrados
[discovery] price-engine-api (latest): 5 archivos encontrados
[discovery] traffic-gate-api (latest): 6 archivos encontrados
[discovery] unified-transaction-configs (v0.0.2): 2 archivos encontrados
[discovery] vendor-stockkeeper-api (0.0.10-stock-consumer): 5 archivos encontrados
[discovery] zenith-keeper (1.1.6): 3 archivos encontrados
[discovery] total: 46 archivos en 10 proyectos
REPO_ROOT: C:\Users\famil\OneDrive\Documentos\Mar\Challenge-Meli-Marcela-Angel
Proyectos resueltos: 10
Archivos fuente descubiertos (tras filtrar navegacion/metadata): 46


## 1. Qué versión de cada proyecto usar (`config/config.yaml`, clave `canonical_sources`)

**El problema:** 4 de los 10 proyectos tienen más de una carpeta de versión (`latest/` y una o más carpetas con un número de versión, por ejemplo `0.2.2/` o `0.0.3-paused-seller/`). Antes de poder leer nada, hay que elegir qué carpeta usar como "la buena" por proyecto — si no, se corre el riesgo de procesar el mismo contenido dos veces y generar preguntas duplicadas.


In [2]:
def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def diff_folders(dir_a, dir_b):
    """Compara 2 carpetas de docs_raw archivo por archivo, por hash (portable, sin
    depender de `diff` de shell)."""
    files_a = {f.relative_to(dir_a).as_posix(): sha256_file(f) for f in dir_a.rglob("*") if f.is_file()}
    files_b = {f.relative_to(dir_b).as_posix(): sha256_file(f) for f in dir_b.rglob("*") if f.is_file()}
    common = set(files_a) & set(files_b)
    different = sorted(p for p in common if files_a[p] != files_b[p])
    only_a = sorted(set(files_a) - set(files_b))
    only_b = sorted(set(files_b) - set(files_a))
    return different, only_a, only_b, len(common)


comparisons = [
    ("catalog-portfolio-api", "0.0.1-test-bench-v2a", "latest"),
    ("payment-promise-gateway", "0.0.10-test-crc-migration", "latest"),
    ("2p-revenue-optimizer-api", "0.2.2", "latest"),
    ("vendor-stockkeeper-api", "0.0.3-paused-seller", "0.0.10-stock-consumer"),
]

summary_rows = []
for project, v_a, v_b in comparisons:
    dir_a = DOCS_RAW_DIR / project / v_a
    dir_b = DOCS_RAW_DIR / project / v_b
    different, only_a, only_b, n_common = diff_folders(dir_a, dir_b)
    summary_rows.append(
        {"proyecto": project, "v_a": v_a, "v_b": v_b, "archivos_comunes": n_common,
         "distintos": len(different), "solo_en_a": len(only_a), "solo_en_b": len(only_b)}
    )
    for p in different:
        print(f"  [{project}] {v_a} vs {v_b} -- distinto: {p}")

pd.DataFrame(summary_rows)

  [payment-promise-gateway] 0.0.10-test-crc-migration vs latest -- distinto: documentationTypes.json
  [2p-revenue-optimizer-api] 0.2.2 vs latest -- distinto: documentationTypes.json
  [2p-revenue-optimizer-api] 0.2.2 vs latest -- distinto: guide/pages/table_data_dictionary/rule_header.md
  [2p-revenue-optimizer-api] 0.2.2 vs latest -- distinto: guide/pages/table_data_dictionary/rule_row.md
  [vendor-stockkeeper-api] 0.0.3-paused-seller vs 0.0.10-stock-consumer -- distinto: documentationTypes.json
  [vendor-stockkeeper-api] 0.0.3-paused-seller vs 0.0.10-stock-consumer -- distinto: guide/_coverpage.md
  [vendor-stockkeeper-api] 0.0.3-paused-seller vs 0.0.10-stock-consumer -- distinto: guide/alerts.md
  [vendor-stockkeeper-api] 0.0.3-paused-seller vs 0.0.10-stock-consumer -- distinto: guide/api.md
  [vendor-stockkeeper-api] 0.0.3-paused-seller vs 0.0.10-stock-consumer -- distinto: guide/important.md


,proyecto,v_a,v_b,archivos_comunes,distintos,solo_en_a,solo_en_b
0,catalog-portfolio-api,0.0.1-test-bench-v2a,latest,10,0,0,0
1,payment-promise-gateway,0.0.10-test-crc-migration,latest,2,1,0,0
2,2p-revenue-optimizer-api,0.2.2,latest,4,3,0,1
3,vendor-stockkeeper-api,0.0.3-paused-seller,0.0.10-stock-consumer,8,5,0,0


**La opción obvia que se descartó: usar siempre `latest/` cuando exista.**
Es lo primero que se le ocurriría a cualquiera. Se descartó porque, al comparar el contenido real, **los datos la contradicen** en al menos 1 de los 4 casos.

**Qué se hizo en su lugar:** una tabla escrita a mano en `config/config.yaml`, donde cada proyecto tiene un campo `reason` obligatorio que explica, con evidencia, por qué se eligió esa versión. Nunca una regla automática basada en el nombre de la carpeta.

- `catalog-portfolio-api` → `latest` (da lo mismo cuál se use; se documenta el motivo).
- `payment-promise-gateway` → `latest` (da lo mismo; se documenta el motivo).
- `2p-revenue-optimizer-api` → `latest` (en este caso sí es realmente la versión más completa).
- `vendor-stockkeeper-api` → `0.0.10-stock-consumer`, **no** `latest` (0.0.10 tiene un endpoint nuevo y un diagrama nuevo que se fueron agregando entre las versiones 0.0.3 → 0.0.6 → 0.0.10; "latest", en cambio, resultó ser una copia vieja y desactualizada).


**Lo que se encontró:** `vendor-stockkeeper-api` es justo el caso donde asumir "latest = más nuevo" hubiera sido un error (arriba se ve que `0.0.3-paused-seller` y `0.0.10-stock-consumer` tienen archivos distintos entre sí — el contenido sí cambió con el tiempo). Al comparar `latest` contra las otras dos, se confirma que es una copia byte por byte de la versión **más vieja** (`0.0.3-paused-seller`), no de la más nueva:


In [3]:
latest_dir = DOCS_RAW_DIR / "vendor-stockkeeper-api" / "latest" / "guide" / "api.md"
old_dir = DOCS_RAW_DIR / "vendor-stockkeeper-api" / "0.0.3-paused-seller" / "guide" / "api.md"
new_dir = DOCS_RAW_DIR / "vendor-stockkeeper-api" / "0.0.10-stock-consumer" / "guide" / "api.md"

print("latest        vs 0.0.3-paused-seller  (deberian coincidir):", sha256_file(latest_dir) == sha256_file(old_dir))
print("latest        vs 0.0.10-stock-consumer (NO deberian coincidir):", sha256_file(latest_dir) == sha256_file(new_dir))

latest        vs 0.0.3-paused-seller  (deberian coincidir): True
latest        vs 0.0.10-stock-consumer (NO deberian coincidir): False


**Un efecto secundario de esta elección, que hubo que resolver en `normalize.py`:** la carpeta elegida (`0.0.10-stock-consumer`) tiene un bug donde comillas y símbolos como `<`/`>` quedan "doble-escapados" (convertidos a código HTML tipo `&#34;`) dentro de bloques de código — algo que no pasa en las otras carpetas del mismo proyecto:


In [4]:
important_0_10 = (DOCS_RAW_DIR / "vendor-stockkeeper-api" / "0.0.10-stock-consumer" / "guide" / "important.md").read_text(encoding="utf-8")
important_0_3 = (DOCS_RAW_DIR / "vendor-stockkeeper-api" / "0.0.3-paused-seller" / "guide" / "important.md").read_text(encoding="utf-8")

print("Ocurrencias de &#34; en 0.0.10-stock-consumer/important.md:", len(re.findall(r"&#34;", important_0_10)))
print("Ocurrencias de &#34; en 0.0.3-paused-seller/important.md: ", len(re.findall(r"&#34;", important_0_3)))

Ocurrencias de &#34; en 0.0.10-stock-consumer/important.md: 546
Ocurrencias de &#34; en 0.0.3-paused-seller/important.md:  0


**Otra opción que se descartó:** usar `0.0.6-ep-restock-man` en vez de `0.0.10` para evitar tener que lidiar con ese bug. Se descartó porque significaría perder el endpoint y el diagrama que solo existen en la versión 0.0.10 — perder esa información vale más que el costo de arreglar un bug de codificación bien acotado (unas pocas líneas en `normalize.py`, ver sección 3).


## 2. Cómo se lee el Markdown (`parse_markdown.py`, `parse_html.py`)

**Qué se decidió:** en vez de escribir un separador de texto propio basado en expresiones regulares (buscar patrones de texto línea por línea), se usa `markdown-it-py`, una librería que convierte el Markdown en un AST — un **árbol de sintaxis**: una representación donde cada título, tabla o bloque de código es un nodo, en vez de texto plano suelto. Se usa su modo más estricto (`commonmark`) con el agregado de soporte para tablas (`.enable("table")`).

Antes de confiar en esta librería para todo el pipeline, se comprobó con un ejemplo chico cómo se comporta realmente (en vez de asumirlo):


In [5]:
from markdown_it import MarkdownIt

md_parser = MarkdownIt("commonmark").enable("table")
sample = '''# Titulo

| a | b |
|---|---|
| 1 | 2 |

```bash
# esto no es un heading, es un comentario bash
echo hi
```

<div class="card"><h3>Card</h3><p>Texto</p></div>
'''

tokens = md_parser.parse(sample)
for t in tokens:
    if t.level == 0:
        print(f"{t.type:20s} map={t.map}")

heading_open         map=[0, 1]
heading_close        map=None
table_open           map=[2, 5]
table_close          map=None
fence                map=[6, 10]
html_block           map=[11, 12]


Esto confirma varias cosas útiles:
- Una tabla (formato GitHub) se convierte en un único "token" de tipo `table_open`, que ya sabe dónde empieza y termina toda la tabla (encabezado y filas) — se puede tratar como una sola unidad, sin lógica extra para detectar sus bordes.
- Un bloque de código (\`\`\`lenguaje ... \`\`\`) se convierte en un único token `fence`, y una línea que empieza con `#` **dentro** de ese bloque (por ejemplo un comentario de bash) no se confunde con un título — la librería ya resuelve esto sola, sin que haya que programar nada para evitarlo.
- Un bloque de HTML embebido (`<div>...</div>`, incluso con etiquetas anidadas adentro) se convierte en un único token que cubre todo ese rango.
- Cada "token" de nivel superior queda marcado como tal (`level == 0`); lo que hay dentro de él (por ejemplo las celdas de una tabla) tiene un nivel mayor y se puede ignorar sin perder nada, porque el token de afuera ya resume todo su contenido.

Gracias a esto no hizo falta escribir a mano un detector de bloques de código/tablas/HTML basado en expresiones regulares — algo más propenso a bugs sutiles con las reglas de Markdown — para lograr el mismo resultado que la librería ya da, verificado.

**Una opción que se descartó: convertir el HTML embebido a su Markdown equivalente.** Reconstruir títulos o énfasis a partir del HTML crudo, el HTML que aparece en `docs_raw` es simple (tarjetas de navegación, secciones colapsables). En su lugar, se usa **BeautifulSoup** (una librería para leer y limpiar HTML) solo para sacar el texto plano (`get_text(" ", strip=True)`) y ese texto se inserta como un párrafo normal, en el mismo lugar donde estaba.

**Los títulos que usaban negrita en Markdown conservaban los asteriscos literalmente**  — por ejemplo `## **Git**` en `demand-forecast-docs/overview.md` aparecía como el título `"**Git**"` en vez de `"Git"`. Se agregó una función (`_clean_heading_title()`) que quita `**negrita**`/`__negrita__` solo de los títulos, no del resto del texto. A propósito **no** se toca la cursiva simple (`*así*` o `_así_`): varios títulos de este corpus son en realidad nombres de columnas con guion bajo (como `rule_custom_field`), y una regla que borrara cursiva de un solo caracter destruiria esos nombres en vez de limpiar formato real.


## 3. Limpieza previa del texto (`normalize.py`)

Antes de convertir el Markdown en un árbol de secciones, se aplican 3 correcciones puntuales. Cada una nace de un bug real y verificado en la documentación.

### 3.1 Títulos mal escritos (sin espacio después del `#`)

`zenith-keeper` tiene títulos como `#Use cases` y `#API FRESH PRODUCTS DOCUMENTATION`, sin espacio después del `#`. Según las reglas de Markdown (CommonMark), eso NO cuenta como un título — quedaría invisible para el árbol de secciones, como si esa parte del documento no tuviera título. Se corrige con una expresión regular que solo actúa **fuera** de los bloques de código (se usa una bandera `in_fence` que se prende y apaga cada vez que aparece una línea que abre o cierra un bloque de código).


In [6]:
from docs2llm.normalize import (
    collapse_duplicate_mermaid_style_lines,
    fix_broken_atx_headings,
    unescape_html_entities_in_code_fences,
)

raw = (DOCS_RAW_DIR / "zenith-keeper" / "1.1.6" / "guide" / "use-cases.md").read_text(encoding="utf-8")
fixed = fix_broken_atx_headings(raw)
print("antes:  ", repr(raw.splitlines()[0]))
print("despues:", repr(fixed.splitlines()[0]))

antes:   '#Use cases'
despues: '# Use cases'


### 3.2 Comillas y símbolos convertidos a código HTML dentro de bloques de código

Como se vio en la sección 1: `vendor-stockkeeper-api/0.0.10-stock-consumer` tiene comillas y los símbolos `<`/`>` convertidos a su versión "escapada" en HTML (`&#34;`, `&#39;`, `&gt;`, `&lt;`) **dentro** de bloques de código JSON — probablemente porque se exportó desde una página ya renderizada. Se usa `html.unescape()` (una función de la librería estándar de Python) para revertir eso, aplicada a **cualquier** archivo, no solo a ese proyecto: si el bug no está presente, la función simplemente no cambia nada, así que es más simple y más seguro que escribir un caso especial tipo "si el proyecto es X, hacé Y":


In [7]:
important = (DOCS_RAW_DIR / "vendor-stockkeeper-api" / "0.0.10-stock-consumer" / "guide" / "important.md").read_text(encoding="utf-8")
important_fixed = unescape_html_entities_in_code_fences(important)
print("Entidades &#34; antes:  ", important.count("&#34;"))
print("Entidades &#34; despues:", important_fixed.count("&#34;"))

Entidades &#34; antes:   546
Entidades &#34; despues: 0


### 3.3 Líneas de estilo repetidas dentro de un mismo diagrama Mermaid

(Mermaid es una forma de escribir diagramas de flujo como texto, dentro de un bloque de código.) `demand-forecast-docs/overview.md` tiene 2 diagramas grandes (entre 250 y 280 líneas cada uno) donde una misma línea de estilo (`style <nodo> fill:...`) aparece 2 veces dentro del mismo diagrama. Se comprobó que no son líneas pegadas por error de copiado — por ejemplo, una de esas líneas repetidas aparece en el renglón 401 y de nuevo en el 520 del mismo bloque, a 120 líneas de distancia — sino la misma instrucción de estilo repetida porque el mismo elemento del diagrama se referencia desde dos ramas distintas del flujo.

Esto se arregla acá, y no en la etapa de deduplicación (`dedup.py`), porque es ruido **dentro de una sola unidad de contenido** (el mismo bloque de código tiene líneas de más), no un duplicado entre dos fragmentos separados del corpus — son dos problemas de naturaleza diferente, y por eso tienen soluciones distintas.


In [8]:
overview_raw = (DOCS_RAW_DIR / "demand-forecast-docs" / "latest" / "guide" / "overview.md").read_text(encoding="utf-8")
overview_collapsed = collapse_duplicate_mermaid_style_lines(overview_raw)

print("Ocurrencias de 'style step__train fill: #fff' antes:  ", overview_raw.count("style step__train fill: #fff"))
print("Ocurrencias de 'style step__train fill: #fff' despues:", overview_collapsed.count("style step__train fill: #fff"))
print(f"Largo del archivo: {len(overview_raw)} -> {len(overview_collapsed)} caracteres ({(1 - len(overview_collapsed)/len(overview_raw)):.1%} de ruido de generacion)")

Ocurrencias de 'style step__train fill: #fff' antes:   4
Ocurrencias de 'style step__train fill: #fff' despues: 2
Largo del archivo: 41345 -> 37654 caracteres (8.9% de ruido de generacion)


## 4. Cómo se lee una especificación OpenAPI (`parse_openapi.py`)

**Qué se decidió:** los archivos `specs/swagger.yaml` (que describen los endpoints de una API, en un formato llamado OpenAPI) tienen su propio lector dedicado — nunca se tratan como si fueran Markdown genérico. Este lector genera una Sección por cada operación (combinación de ruta + método HTTP) y una por cada "schema" (la forma de un objeto de datos), como texto compacto y legible — no el YAML crudo.

**Resolver referencias (`$ref`) sin asumir dónde van a estar.** Un `$ref` es un puntero dentro del mismo archivo YAML: algo como "buscar la definición a tal otro lugar del documento". Se comprobó que los 3 archivos `swagger.yaml` usan convenciones distintas: `traffic-gate-api` y `zenith-keeper` guardan sus definiciones en el lugar estándar (`components/schemas/*`), pero `price-engine-api` las guarda en un lugar no estándar (`$ref: '#/models/Zones'`). El código que sigue esos punteros camina el documento ya leído, paso a paso, siguiendo literalmente la ruta indicada — sin asumir que siempre va a empezar en "components". Así funciona igual para las dos convenciones, sin necesitar un caso especial por proyecto:


In [9]:
from docs2llm.parse_openapi import parse_openapi_sections

for project in ["price-engine-api", "traffic-gate-api", "zenith-keeper"]:
    swagger_path = sources[project].path / "specs" / "swagger.yaml"
    sections = parse_openapi_sections(swagger_path.read_text(encoding="utf-8"))
    print(f"{project}: {len(sections)} secciones (paths + schemas)")

  [parse_openapi] 1 operaciones, 2 schemas
price-engine-api: 3 secciones (paths + schemas)
  [parse_openapi] 8 operaciones, 5 schemas
traffic-gate-api: 13 secciones (paths + schemas)
  [parse_openapi] 14 operaciones, 21 schemas
zenith-keeper: 35 secciones (paths + schemas)


Ejemplo real de esa resolución no estándar en `price-engine-api`: `Zones` y `RequestError` se resuelven a través de `#/models/...`, fuera del lugar habitual (`components/`):


In [10]:
price_sections = parse_openapi_sections((sources["price-engine-api"].path / "specs" / "swagger.yaml").read_text(encoding="utf-8"))
for s in price_sections:
    print(f"--- {' > '.join(s.section_path)} ---")
    print(s.blocks[0].text)
    print()

  [parse_openapi] 1 operaciones, 2 schemas
--- swagger.yaml > paths > GET /sites/{site_id}/sellers/{seller_id}/services/{service_id}/coverage/v1 ---
Endpoint: GET /sites/{site_id}/sellers/{seller_id}/services/{service_id}/coverage/v1
Operation ID: GetActiveCoverageAdjustmentsBySeller
Summary: Obtención del estado del ajuste de configuracion de cobertura del seller
Description: **Responsabilidad**<br/>
Devolver un array con las zonas del seller moderado por cobertura.<br/>

**Limitaciones**<br/>
Si el seller no tiene ajustes de configuracion de cobertura activos, este endpoint devolverá un array vacío.<br/>
Parameters:
  - site_id (in: path, type: string) enum=['MLA', 'MLB', 'MLC', 'MCO', 'MLU', 'MPE', 'MLM', 'MEC'] [required]: ID del Site.
  - service_id (in: path, type: integer) [required]: ID servicio relacionado al seller.
  - seller_id (in: path, type: integer) [required]: ID del seller que genera el token.
  - Accept-Version (in: header, type: string) [required]: Versión del endpo

**Cuando se describe un schema con propiedades anidadas (objetos dentro de objetos), solo se detalla un nivel**  — si una propiedad es a su vez un objeto con sus propias propiedades, esas no se listan, solo se muestra su tipo. Se aceptó esta limitación porque los 3 archivos `swagger.yaml` de `docs_raw` no tienen más de 1 o 2 niveles de anidamiento en la práctica.

**Nota:** al imprimir texto con tildes en la consola de Windows pueden aparecer caracteres raros como `�` (por ejemplo "Versi�n" en vez de "Versión"). Es solo un problema de cómo la consola de Windows muestra el texto por defecto — el pipeline en sí siempre lee y escribe archivos indicando explícitamente `encoding="utf-8"`, así que esto nunca afecta a los archivos `.jsonl` que genera, solo a cómo se ve un mensaje de depuración en una terminal puntual.


## 5. Cómo se dividen los documentos en fragmentos (`chunking.py`) — `tiktoken`

`chunking.py` transforma las secciones estructuradas del documento en chunks optimizados para recuperación semántica. Respeta los límites naturales del contenido (párrafos, tablas, listas y código), evita fragmentar información salvo en casos excepcionales, fusiona secciones demasiado pequeñas para mejorar la densidad informativa y mantiene las especificaciones OpenAPI como unidades independientes por su valor semántico. Además, identifica bloques repetidos para facilitar la deduplicación posterior y asigna a cada chunk un **content_type dominante** basado en el bloque de mayor tamaño, permitiendo distinguir si el contenido principal es prosa, código, tabla o especificación de API. El resultado es un corpus más coherente, mejor etiquetado y mucho más útil para sistemas RAG que un chunking ingenuo basado únicamente en longitud de texto.


`chunking.py` y `config.py`: el conteo de tokens usa `tiktoken` directamente. La celda de código que sigue es evidencia válida de que el empaquetado de chunks funciona, los valores de `n_tokens` que reporta son conteos de `tiktoken`.

In [11]:
from docs2llm.parse_markdown import parse_markdown_sections
from docs2llm.normalize import normalize_markdown_text
from docs2llm.parse_openapi import parse_openapi_sections
from docs2llm.chunking import chunk_sections

chunks_sin_dedup_de_bloque = []
for f in files:
    text = f.absolute_path.read_text(encoding="utf-8")
    if f.file_kind == "markdown":
        secs = parse_markdown_sections(normalize_markdown_text(text), fallback_title=f.absolute_path.stem)
    else:
        secs = parse_openapi_sections(text)
    chunks_sin_dedup_de_bloque.extend(chunk_sections(secs, pipeline_cfg.chunking))

n_tokens = [c.n_tokens for c in chunks_sin_dedup_de_bloque]
print(f"Total archivos procesados: {len(files)}")
print(f"Total chunks (sin deteccion de duplicados a nivel de bloque, ver seccion 6): {len(chunks_sin_dedup_de_bloque)}")
print(f"Max tokens observado: {max(n_tokens)} (limite configurado: {pipeline_cfg.chunking.max_tokens})")
print(f"Promedio de tokens/chunk: {sum(n_tokens)/len(n_tokens):.1f}")
pd.Series(Counter(c.content_type for c in chunks_sin_dedup_de_bloque), name="chunks").to_frame()

  [parse_markdown] 1 secciones, 3 bloques
  [chunking] 1 secciones -> 1 chunks
  [parse_markdown] 1 secciones, 3 bloques
  [chunking] 1 secciones -> 1 chunks
  [parse_markdown] 1 secciones, 3 bloques
  [chunking] 1 secciones -> 1 chunks
  [parse_markdown] 10 secciones, 23 bloques
  [chunking] 10 secciones -> 7 chunks
  [parse_markdown] 19 secciones, 59 bloques
  [chunking] 19 secciones -> 10 chunks
  [parse_markdown] 10 secciones, 15 bloques
  [chunking] 10 secciones -> 6 chunks
  [parse_markdown] 11 secciones, 13 bloques
  [chunking] 11 secciones -> 6 chunks
  [parse_markdown] 13 secciones, 20 bloques
  [chunking] 13 secciones -> 7 chunks
  [parse_markdown] 7 secciones, 17 bloques
  [chunking] 7 secciones -> 4 chunks
  [parse_markdown] 13 secciones, 21 bloques
  [chunking] 13 secciones -> 6 chunks
  [parse_markdown] 5 secciones, 10 bloques
  [chunking] 5 secciones -> 2 chunks
  [parse_markdown] 21 secciones, 28 bloques
  [chunking] 21 secciones -> 14 chunks
  [parse_markdown] 6 seccio

,chunks
table,24
prose,106
code,71
api_spec,51


## 6. Deduplicación en dos capas (`dedup.py`)

Dos preguntas necesitan trato distinto: "¿este chunk es un duplicado EXACTO de otro?" (hash, sección 6.1) y "¿este chunk dice CASI lo mismo que otro, aunque el texto haya cambiado?" (embeddings, sección 6.2). El diseño de esta etapa tiene las dos capas desde el arranque: la primera es barata y determinista, y cubre la mayoría de la duplicación real encontrada en `docs_raw`; la segunda cubre el caso donde el contenido evolucionó entre versiones sin dejar de ser, en esencia, lo mismo.

### 6.1 Detección en dos pasadas (hash exacto a nivel de bloque)

**Primer intento: comparar por hash el texto final de cada chunk ya armado.** Al correr `build-corpus` por primera vez contra los archivos reales, el resultado fue **0 duplicados encontrados**.

**Diagnóstico:** se buscó a mano, dentro de `corpus.jsonl`, el texto `brand_not_found`. Aparecía en varios chunks, pero cada uno mezclado con texto **distinto** alrededor — el armado de chunks (`chunking.py`) estaba juntando el texto repetido con bloques vecinos diferentes en cada archivo, así que el chunk final nunca terminaba siendo idéntico entre una aparición y otra, aunque la parte repetida en el medio sí lo fuera. Comparar por hash el chunk ya armado es, en la práctica, ciego a este tipo de casos.

**Detectar duplicados en dos pasadas.** Ahora `corpus_build.py` primero recorre **todos** los archivos y los separa en secciones y bloques (sin armar los chunks finales todavía), y calcula, mirando el corpus completo, qué bloques individuales tienen el mismo contenido exacto repetido en más de un archivo. Esa lista de bloques repetidos se le pasa después a `chunk_sections()`, que trata a cualquier bloque marcado como "ya visto en otro archivo" como una unidad aislada que nunca se mezcla con sus vecinos — el mismo tratamiento que ya se le daba a un bloque demasiado grande o a una sección de OpenAPI.

In [12]:
from docs2llm.corpus_build import build_corpus_rows

rows, split_report, near_duplicate_pairs = build_corpus_rows(sources, pipeline_cfg)
duplicate_rows = [r for r in rows if r.duplicate_of is not None]

print(f"Total chunks (CON deteccion de duplicados a nivel de bloque): {len(rows)}")
print(f"  (antes del fix: {len(chunks_sin_dedup_de_bloque)} chunks, 0 duplicados detectados)")
print(f"Marcados como duplicado EXACTO: {len(duplicate_rows)} en {len({r.duplicate_of for r in duplicate_rows})} clusters distintos")

[build-corpus] descubriendo archivos en docs_raw/...
[discovery] 2p-revenue-optimizer-api (latest): 4 archivos encontrados
[discovery] catalog-portfolio-api (latest): 8 archivos encontrados
[discovery] demand-forecast-docs (latest): 4 archivos encontrados
[discovery] order-workflow-api (5.6.2-hotfix-1): 8 archivos encontrados
[discovery] payment-promise-gateway (latest): 1 archivos encontrados
[discovery] price-engine-api (latest): 5 archivos encontrados
[discovery] traffic-gate-api (latest): 6 archivos encontrados
[discovery] unified-transaction-configs (v0.0.2): 2 archivos encontrados
[discovery] vendor-stockkeeper-api (0.0.10-stock-consumer): 5 archivos encontrados
[discovery] zenith-keeper (1.1.6): 3 archivos encontrados
[discovery] total: 46 archivos en 10 proyectos

[build-corpus] primera pasada: parseando 46 archivos y detectando bloques duplicados...
[build-corpus] leyendo docs_raw/2p-revenue-optimizer-api/latest/guide/pages/table_data_dictionary/rule_custom_field.md
  [parse_m

c:\Users\famil\OneDrive\Documentos\Mar\Challenge-Meli-Marcela-Angel\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


[dedup] 17 pares casi-duplicados encontrados (umbral=0.9, modelo all-MiniLM-L6-v2)
[dedup] 45 documentos agrupados en 39 componentes conexas

[build-corpus] asignando splits para 10 proyectos...
[build-corpus] listo: 334 filas finales

Total chunks (CON deteccion de duplicados a nivel de bloque): 334
  (antes del fix: 252 chunks, 0 duplicados detectados)
Marcados como duplicado EXACTO: 37 en 16 clusters distintos


**Un ejemplo real, verificado línea por línea:** el fragmento de JSON `{"error": "brand_not_found", ...}` aparece tanto en `brands.md` (que queda como la versión "canónica", la primera encontrada) como en `api-reference/README.md` — y este último queda correctamente marcado con `duplicate_of` apuntando al primero:


In [13]:
brand_not_found_rows = [r for r in rows if "brand_not_found" in r.text and len(r.text) < 300]
for r in brand_not_found_rows:
    print(f"{r.source_path}\n  n_tokens={r.n_tokens}  is_low_signal={r.is_low_signal}  duplicate_of={r.duplicate_of}")

docs_raw/catalog-portfolio-api/latest/guide/api-reference/brands.md
  n_tokens=41  is_low_signal=False  duplicate_of=None
docs_raw/catalog-portfolio-api/latest/guide/api-reference/README.md
  n_tokens=41  is_low_signal=False  duplicate_of=docs_raw/catalog-portfolio-api/latest/guide/api-reference/brands.md#49


**Un efecto secundario esperado (no un error): la cantidad de chunks marcados `is_low_signal` sube bastante respecto a la sección 5.** Al aislar los fragmentos duplicados que antes quedaban "escondidos" dentro de chunks más grandes (como el botón roto "Download X Template", que es puro ruido de diseño sin contenido real), esos fragmentos ahora se miden por su propio tamaño — y como son chicos, correctamente quedan por debajo del umbral mínimo de tokens informativos. Esto es una mejora, no un problema: hace visible que ese contenido es ruido, en vez de dejarlo disimulado dentro de un chunk más grande que parecía tener sustancia.


In [14]:
print(f"Chunks is_low_signal (con dedup de bloque, seccion 6): {sum(r.is_low_signal for r in rows)}")
print(f"Chunks is_low_signal (sin dedup de bloque, seccion 5):  {sum(c.is_low_signal for c in chunks_sin_dedup_de_bloque)}")

Chunks is_low_signal (con dedup de bloque, seccion 6): 45
Chunks is_low_signal (sin dedup de bloque, seccion 5):  3


### 6.2 Por qué además hace falta una segunda capa: duplicados aproximados por embeddings

**El caso que el hash no puede ver:** en `2p-revenue-optimizer-api`, la tabla de reglas de negocio cambia de columnas entre versiones (`created_date`→`date`, `productive_date`→`version`) pero sigue describiendo, en esencia, lo mismo. El hash exacto (aún con dos pasadas, sección 6.1) da hashes distintos para las dos versiones — es ciego a contenido que cambió de texto pero no de significado.

**Qué se hizo:** una segunda capa que compara embeddings semánticos (`sentence-transformers`, modelo `all-MiniLM-L6-v2`) con similitud coseno (`find_near_duplicate_pairs` en `dedup.py`). Dos chunks del mismo `content_type` con similitud >= `near_duplicate_threshold` (0.9 en producción) se marcan como casi-duplicados. A diferencia de un duplicado exacto, esto NO se marca con `duplicate_of` (el texto no es literalmente idéntico, sería engañoso llamarlo así) — pero sí alimenta las mismas componentes conexas que usa `splitting.py` (sección 7), por la misma razón que un duplicado exacto: si quedara repartido entre train y test, el modelo vería en test una versión casi idéntica de algo que ya vio en train.

`near_duplicate_pairs`, ya calculado arriba (al correr `build_corpus_rows`, sección 6.1), tiene el resultado real de esta capa sobre el corpus completo:

In [15]:
print(f"Pares casi-duplicados encontrados: {len(near_duplicate_pairs)} (umbral={pipeline_cfg.dedup.near_duplicate_threshold})")

rows_by_id = {r.id: r for r in rows}
for id_a, id_b in near_duplicate_pairs:
    if "2p-revenue-optimizer-api" in id_a:
        print(f"\n{id_a}\n{id_b}")
        print("texto A:", rows_by_id[id_a].text.replace("\n", " ")[:160])
        print("texto B:", rows_by_id[id_b].text.replace("\n", " ")[:160])

Pares casi-duplicados encontrados: 17 (umbral=0.9)

docs_raw/2p-revenue-optimizer-api/latest/guide/pages/table_data_dictionary/rule_header.md#0
docs_raw/2p-revenue-optimizer-api/latest/guide/pages/table_data_dictionary/rule_row.md#0
texto A: The `rule_header` table stores , receiving data from PME.
texto B: The `rule_row` table stores , receiving data from PME.


**Cómo se calibró el umbral 0.9 (no es un valor de la literatura, es empírico contra este corpus).** El enfoque general: embeddings + similitud coseno para deduplicación semántica de un corpus de entrenamiento. El número concreto se midió directo contra `docs_raw`, comparando la similitud real de un par que SÍ es casi-duplicado contra la de un par sin relación:

In [16]:
from docs2llm import dedup as dedup_module
from sklearn.metrics.pairwise import cosine_similarity

model = dedup_module._embedding_model()

par_relacionado_a, par_relacionado_b = next((a, b) for a, b in near_duplicate_pairs if "2p-revenue-optimizer-api" in a)
texto_relacionado_a = rows_by_id[par_relacionado_a].text
texto_relacionado_b = rows_by_id[par_relacionado_b].text
texto_no_relacionado_a = next(r.text for r in rows if r.project == "demand-forecast-docs" and r.content_type == "prose")
texto_no_relacionado_b = next(r.text for r in rows if r.project == "traffic-gate-api" and r.content_type == "prose")

embeddings = model.encode(
    [texto_relacionado_a, texto_relacionado_b, texto_no_relacionado_a, texto_no_relacionado_b],
    convert_to_numpy=True,
)
sim = cosine_similarity(embeddings)
print(f"Similitud del par casi-duplicado real (2p-revenue-optimizer-api): {sim[0][1]:.3f}")
print(f"Similitud de un par SIN relacion (proyectos distintos):          {sim[2][3]:.3f}")

Similitud del par casi-duplicado real (2p-revenue-optimizer-api): 0.906
Similitud de un par SIN relacion (proyectos distintos):          0.036


**Un hallazgo que ajustó el diseño: los chunks `api_spec` dan falsos positivos a este umbral.** El formato de `parse_openapi.py` es muy fijo y repetido (`"Endpoint: ..."`, `"Operation ID: ..."`, `"Parameters:"`...) — eso hace que operaciones o schemas GENUINAMENTE DISTINTOS embedan con similitud alta solo por compartir plantilla, no contenido real repetido. Se verificó comparando el resultado CON y SIN excluir `api_spec` de esta capa (ver `_CONTENT_TYPES_SIN_NEAR_DUPLICATE_CHECK` en `dedup.py`):

In [17]:
original_exclusion = dedup_module._CONTENT_TYPES_SIN_NEAR_DUPLICATE_CHECK
dedup_module._CONTENT_TYPES_SIN_NEAR_DUPLICATE_CHECK = set()  # temporalmente sin exclusiones, solo para medir el efecto
pares_sin_exclusion = dedup_module.find_near_duplicate_pairs(rows, pipeline_cfg.dedup.near_duplicate_threshold)
dedup_module._CONTENT_TYPES_SIN_NEAR_DUPLICATE_CHECK = original_exclusion  # se restaura de inmediato

pares_api_spec = [(a, b) for a, b in pares_sin_exclusion if rows_by_id[a].content_type == "api_spec"]
print(f"Pares a umbral 0.9 SIN excluir api_spec: {len(pares_sin_exclusion)}")
print(f"  de los cuales son api_spec (plantilla compartida, no duplicado real): {len(pares_api_spec)}")
print(f"Pares finales (funcion real, CON la exclusion): {len(near_duplicate_pairs)}")

if pares_api_spec:
    ej_a, ej_b = pares_api_spec[0]
    print(f"\nEjemplo de par api_spec excluido -- misma plantilla, contenido distinto:\n--- {ej_a} ---")
    print(rows_by_id[ej_a].text[:280])
    print(f"--- {ej_b} ---")
    print(rows_by_id[ej_b].text[:280])

[dedup] 24 pares casi-duplicados encontrados (umbral=0.9, modelo all-MiniLM-L6-v2)
Pares a umbral 0.9 SIN excluir api_spec: 24
  de los cuales son api_spec (plantilla compartida, no duplicado real): 7
Pares finales (funcion real, CON la exclusion): 17

Ejemplo de par api_spec excluido -- misma plantilla, contenido distinto:
--- docs_raw/zenith-keeper/1.1.6/specs/swagger.yaml#0 ---
Endpoint: POST /api/v1/fresh-products/inboundorder
Operation ID: addBachOfProductsToStock
Summary: Insert batch in fulfillment warehouse
Request body [required]:
  content-type: application/json
  -> schema 'InsertBatchRequestDTO' (ver chunk de schemas para el detalle completo)
R
--- docs_raw/zenith-keeper/1.1.6/specs/swagger.yaml#1 ---
Endpoint: PUT /api/v1/fresh-products/inboundorder
Operation ID: updateBachOfProductsToStock
Summary: Update batch in fulfillment warehouse
Request body [required]:
  content-type: application/json
  -> schema 'InsertBatchRequestDTO' (ver chunk de schemas para el detalle comple

Con la exclusión de `api_spec`, el umbral 0.9 deja los pares mostrados arriba como casi-duplicados reales. Estos pares alimentan las mismas componentes conexas que los duplicados exactos (ver `file_duplicate_components(rows, near_duplicate_pairs=...)`, usado también en la sección 7): ningún documento que comparta contenido casi-duplicado con otro queda repartido entre train y test.

## 7. Como se dividen los datos en train/val/test (`splitting.py`)

**Por que se divide por documento y no por fragmento (chunk):** dividir a nivel de fragmento podria dejar, por ejemplo, dos fragmentos casi identicos del **mismo** archivo en splits distintos -- una fuga de datos directa (el modelo "veria" en test algo casi igual a lo que ya vio en train).

**Por que se divide por grupo de documentos conectados (componente conexa) y no por archivo individual:** como se encontro en `catalog-portfolio-api` (varios archivos comparten fragmentos exactos), esos archivos tienen que quedar siempre en el mismo split -- si no, el mismo fragmento terminaria apareciendo en train y en test al mismo tiempo.

**Por que la cuota se reparte por CANTIDAD DE CHUNKS y no por cantidad de documentos:** los documentos de este corpus varian mucho en tamano (de 1 a mas de 50 chunks) -- repartir "por cabeza" ignora ese peso por completo.

**Por que el balanceo es GLOBAL sobre todo el corpus, y no una cuota 60/20/20 recalculada dentro de cada proyecto:** la primera version de este modulo si hacia el balanceo por proyecto, pero eso resulto en un problema real, que se encontro corriendo el pipeline completo: cada proyecto, evaluado de forma aislada, manda "razonablemente" su documento mas pesado a la cuota mas grande (train) -- pero con 10 proyectos haciendo eso al mismo tiempo, sin que ninguno sepa cuanto ya absorbio train en los proyectos anteriores, el train agregado terminaba en ~74% del corpus real (medido en chunks), muy por encima del 60% buscado (ver la celda de abajo para los numeros reales, antes/despues). Por eso `assign_splits` ahora corre un unico algoritmo LPT (longest-processing-time-first, un heuristico clasico de balanceo de carga) sobre TODOS los grupos de documentos del corpus a la vez: ordena los grupos de mayor a menor peso (en chunks) y asigna cada uno, de a uno, al split que en ESE momento (con el estado acumulado de TODO lo ya asignado) este mas lejos de su cuota objetivo.

**El costo de este cambio:** ya no se garantiza que cada proyecto tenga representacion en los 3 splits (antes si, por diseno) -- queda auditado con warnings explicitos en `split_report.json` cuando un proyecto pierde representacion en val o test (ver la celda de abajo).

**Por que se escribio un repartidor propio, en vez de usar herramientas ya armadas de scikit-learn (como `GroupShuffleSplit` o `StratifiedGroupKFold`):** el comportamiento de esas herramientas frente a grupos de peso muy dispar (un documento de 1 chunk junto a uno de 51) es dificil de predecir y de explicar en vivo.


In [18]:
from docs2llm import dedup as dedup_module

components = dedup_module.file_duplicate_components(rows, near_duplicate_pairs=near_duplicate_pairs)
cat_docs = sorted({r.doc_id for r in rows if r.project == "catalog-portfolio-api"})
doc_split = {r.doc_id: r.split for r in rows}

by_component = defaultdict(list)
for d in cat_docs:
    by_component[components[d]].append(d)

for comp, docs in by_component.items():
    if len(docs) > 1:
        print("Componente compartido (garantizado en el MISMO split):")
        for d in docs:
            print(f"  [{doc_split[d]}] {d}")

[dedup] 45 documentos agrupados en 39 componentes conexas
Componente compartido (garantizado en el MISMO split):
  [train] docs_raw/catalog-portfolio-api/latest/guide/api-reference/README.md
  [train] docs_raw/catalog-portfolio-api/latest/guide/api-reference/brands.md
  [train] docs_raw/catalog-portfolio-api/latest/guide/brand-management/README.md
  [train] docs_raw/catalog-portfolio-api/latest/guide/setup/README.md


Los 4 archivos de arriba quedan conectados de forma **transitiva** (no todos comparten contenido directamente entre sí, sino a través de otros) por 3 fragmentos exactos distintos — y esa es justo la razón por la que hace falta una estructura que agrupe elementos conectados (un "union-find", que agrupa cosas conectadas entre sí aunque sea indirectamente) en vez de simplemente comparar cada archivo contra cada otro archivo:


In [19]:
target_docs = set(by_component[components[cat_docs[0]]])  # el componente compartido de arriba, cualquier archivo del grupo
# (recalculado explicitamente para el grupo real, no asumido)
target_docs = {d for d, docs in [(d, by_component[components[d]]) for d in cat_docs] if len(docs) > 1}

by_hash = defaultdict(list)
for r in rows:
    if r.doc_id in target_docs:
        by_hash[r.content_hash_sha256].append(r)

for h, rs in by_hash.items():
    docs_involved = {r.doc_id for r in rs}
    if len(docs_involved) > 1:
        print("Fragmento EXACTO compartido entre:")
        for d in docs_involved:
            print(f"   {d}")
        print(f"   texto: {rs[0].text[:100]!r}\n")

Fragmento EXACTO compartido entre:
   docs_raw/catalog-portfolio-api/latest/guide/brand-management/README.md
   docs_raw/catalog-portfolio-api/latest/guide/api-reference/brands.md
   texto: '```bash\ncurl -X PUT "https://api.mp.internal.retailflow.com/catalog/brands/123/activate"\n```'

Fragmento EXACTO compartido entre:
   docs_raw/catalog-portfolio-api/latest/guide/api-reference/README.md
   docs_raw/catalog-portfolio-api/latest/guide/api-reference/brands.md
   texto: '```json\n{\n  "error": "brand_not_found",\n  "message": "Brand with ID 123 was not found",\n  "status": '

Fragmento EXACTO compartido entre:
   docs_raw/catalog-portfolio-api/latest/guide/api-reference/README.md
   docs_raw/catalog-portfolio-api/latest/guide/brand-management/README.md
   docs_raw/catalog-portfolio-api/latest/guide/setup/README.md
   texto: '**Available Items:**'



### Como quedo realmente cada proyecto, y el balance GLOBAL medido en chunks

`split_report.json` muestra la composicion que realmente se logro -- por proyecto (documentos y chunks) y, mas abajo, el numero agregado que de verdad importa: el porcentaje real de todo el corpus en cada split, medido en chunks.


In [20]:
docs_df = pd.DataFrame({p: d["conteo_por_split"] for p, d in split_report.items()}).T
docs_df["n_documentos_unicos"] = [split_report[p]["n_documentos_unicos"] for p in docs_df.index]
docs_df["n_documentos_totales"] = [split_report[p]["n_documentos_totales"] for p in docs_df.index]
print("Conteo por proyecto, en CANTIDAD DE DOCUMENTOS/GRUPOS (no en chunks):")
docs_df

chunks_df = pd.DataFrame({p: d["chunks_por_split"] for p, d in split_report.items()}).T
print("\nConteo por proyecto, en CANTIDAD DE CHUNKS (la unidad que de verdad ve un modelo):")
chunks_df

# El numero que de verdad importa: el balance GLOBAL medido en chunks (no
# promediado por proyecto) -- es lo que el algoritmo LPT de splitting.py
# esta optimizando de verdad.
chunks_por_split_global = Counter()
for d in split_report.values():
    for split, n in d["chunks_por_split"].items():
        chunks_por_split_global[split] += n
total_chunks = sum(chunks_por_split_global.values())

print(f"\nBalance GLOBAL real del split, medido en chunks (total={total_chunks}):")
for split in ("train", "val", "test"):
    n = chunks_por_split_global[split]
    print(f"  {split:5s}: {n:4d} chunks  ({n/total_chunks:6.1%})")


Conteo por proyecto, en CANTIDAD DE DOCUMENTOS/GRUPOS (no en chunks):

Conteo por proyecto, en CANTIDAD DE CHUNKS (la unidad que de verdad ve un modelo):

Balance GLOBAL real del split, medido en chunks (total=334):
  train:  200 chunks  ( 59.9%)
  val  :   67 chunks  ( 20.1%)
  test :   67 chunks  ( 20.1%)


In [21]:
# Invariante de fuga: ningun hash de contenido duplicado aparece en 2 splits distintos.
split_by_hash = {}
leaks = 0
for r in rows:
    if r.content_hash_sha256 in split_by_hash and split_by_hash[r.content_hash_sha256] != r.split:
        leaks += 1
    else:
        split_by_hash[r.content_hash_sha256] = r.split

print(f"Fugas de datos detectadas entre splits (deberia ser 0): {leaks}")
print(f"Splits presentes: {sorted({r.split for r in rows})}")

Fugas de datos detectadas entre splits (deberia ser 0): 0
Splits presentes: ['test', 'train', 'val']


Esta misma regla (que no haya fugas de datos entre splits) corre también como un test automático, en `tests/test_split_no_leakage.py::test_integracion_corpus_real_sin_fugas_entre_splits` — contra el corpus real completo, no solo contra casos de prueba armados a mano.


## 8. Cómo se generan las preguntas y respuestas (`qa_templates.py`, `qa_llm_generate.py`, `qa_grounding.py`, `qa_build.py`)

**El enfoque: combinar dos métodos.** Para contenido estructurado (tablas, especificaciones OpenAPI) se usan plantillas de texto; para prosa y reglas de negocio, un modelo de lenguaje (LLM).

**Plantillas (`qa_templates.py`).** Trabajan sobre el texto que ya quedó armado en `corpus.jsonl`, sin volver a leer `docs_raw/`: como `parse_openapi.py` ya escribe un formato de texto fijo y conocido (`"Endpoint: ..."`, `"Schema: ..."`), se usan expresiones regulares dirigidas exactamente a ese formato para volver a sacar los campos. La respuesta está garantizada de estar bien fundamentada (grounded) por cómo está construido el proceso: la respuesta ES literalmente el contenido de una celda de tabla o de un campo de la especificación.

**LLM (`qa_llm_generate.py`).** Usa "tool use" (una forma de la API de Claude donde el modelo devuelve datos en un formato fijo, en vez de texto libre) en lugar de pedirle al modelo que devuelva JSON como texto suelto. Las instrucciones que recibe el modelo (el "system prompt") exigen que `evidence_span` sea una cita **literal** del fragmento fuente, y que el modelo devuelva una lista vacía en vez de forzar una pregunta floja si el fragmento no tiene contenido sustancioso.

**Control de calidad (`qa_grounding.py`), aplicado a todos los candidatos generados:**
1. `check_evidence`: la cita (`evidence_span`) tiene que ser una subcadena literal del fragmento fuente.
2. `check_length`: la pregunta debe tener entre 8 y 300 caracteres, y la respuesta no puede estar vacía.
3. `find_near_duplicate_question_indices`: descarta preguntas casi idénticas dentro del mismo proyecto.

**Límite anti-volumen (`qa_build.py`):** un tope de `max_pairs_per_document` pares por **documento**, no por fragmento — sin este límite, un archivo con muchas secciones estructuradas (como `zenith-keeper/specs/swagger.yaml`, con ~35 secciones) terminaría dominando el dataset final solo por su volumen, no porque tenga más señal útil real.

**Los fragmentos duplicados o de bajo contenido (`is_low_signal`) no generan preguntas vía LLM:** un fragmento marcado como duplicado de otro se salta directamente; un fragmento de prosa marcado `is_low_signal=True` nunca se le manda al LLM — pero sí puede generar una pregunta por plantilla si es una tabla o especificación chica (una respuesta correcta de 3 palabras sigue siendo una respuesta correcta).

In [22]:
from docs2llm.qa_templates import generate_template_qa

# Replica la MISMA condicion que qa_build._generate_candidates_for_row: un chunk
# marcado duplicate_of se salta (ya hay una pregunta equivalente sobre su canonico).
# Sin este filtro el conteo da 252 en vez de 248 -- se detecto esta discrepancia
# corriendo esta celda y comparandola contra qc_report.json antes de darla por buena.
candidatos_template = 0
for r in rows:
    if r.duplicate_of is not None:
        continue
    if r.content_type in ("table", "api_spec"):
        candidatos_template += len(generate_template_qa(r.text, r.content_type, r.section_path))

print(f"Candidatos generados por templates (deterministico, corre siempre): {candidatos_template}")

Candidatos generados por templates (deterministico, corre siempre): 248


### Verificación sobre el `qa.jsonl` ya generado (incluye los pares hechos con LLM)

La parte que usa el LLM no se vuelve a correr cada vez que se ejecuta este notebook (costaría tiempo y dinero, y depende de tener configurada la clave `ANTHROPIC_API_KEY`). En cambio, se lee y verifica el `qa.jsonl` que ya quedó guardado en el repo.


In [23]:
qa_path = REPO_ROOT / "data" / "processed" / "qa.jsonl"

if qa_path.exists():
    qa_rows = [json.loads(line) for line in qa_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"qa.jsonl: {len(qa_rows)} filas")
    print(f"  por metodo: {dict(Counter(r['generation_method'] for r in qa_rows))}")
    print(f"  qc_passed=True: {sum(r['qc_passed'] for r in qa_rows)}/{len(qa_rows)}")

    # Invariante de grounding sobre el archivo COMPLETO
    corpus_by_id = {r.id: r for r in rows}
    def collapse(s):
        return re.sub(r"\s+", " ", s).strip()

    mismatches = 0
    split_mismatches = 0
    for qr in qa_rows:
        chunk = corpus_by_id.get(qr["source_chunk_id"])
        if chunk is None or collapse(qr["evidence_span"]) not in collapse(chunk.text):
            mismatches += 1
        elif qr["split"] != chunk.split:
            split_mismatches += 1
    print(f"  mismatches de evidence_span (deberia ser 0): {mismatches}")
    print(f"  filas con split distinto al de su chunk de origen (deberia ser 0): {split_mismatches}")
else:
    print("qa.jsonl no generado todavia en este checkout -- correr `python -m docs2llm build-qa` primero.")
    qa_rows = []

qa.jsonl: 248 filas
  por metodo: {'llm': 195, 'template': 53}
  qc_passed=True: 248/248
  mismatches de evidence_span (deberia ser 0): 0
  filas con split distinto al de su chunk de origen (deberia ser 0): 0


**Revisión manual del contenido (no solo reglas automáticas):** se comprobó a mano que el proyecto `order-workflow-api` genera preguntas correctas sobre automatización de releases y flujo de git (ninguna pregunta asume, por el nombre, que se trata de órdenes de compra de e-commerce), y que `zenith-keeper` genera preguntas correctas sobre su dominio real (inventario de productos frescos). Esto confirma que el modelo no arrastró la confusión de nombres.


In [24]:
if qa_rows:
    for project in ["order-workflow-api", "zenith-keeper"]:
        print(f"--- {project} ---")
        for r in [q for q in qa_rows if q["project"] == project][:3]:
            print(f"  Q: {r['question']}")
            print(f"  A: {r['answer']}")
        print()

--- order-workflow-api ---
  Q: ¿Qué tareas realiza Workflow API para simplificar y estandarizar el flujo de desarrollo de aplicaciones integradas en Release Process?
  A: Valida el modelo de branching, protege las ramas importantes (mínimos aprobadores y checks requeridos), crea backports para mantener las ramas sincronizadas, y crea versiones automáticas basadas en una configuración tipo cron.
  Q: ¿Qué tipo de base de datos se utiliza para almacenar la configuración del workflow?
  A: Una base de datos MySQL
  Q: ¿Para qué se utiliza el servicio Jobs?
  A: Para crear automáticamente versiones basadas en un scheduler tipo cron.

--- zenith-keeper ---
  Q: ¿Qué hace el endpoint `POST /api/v1/fresh-products/inboundorder`?
  A: Insert batch in fulfillment warehouse
  Q: ¿Qué hace el endpoint `PUT /api/v1/fresh-products/inboundorder`?
  A: Update batch in fulfillment warehouse
  Q: ¿Qué hace el endpoint `GET /api/v1/fresh-products`?
  A: Get complete products list

